# 01 — Dataset Preparation

**Pipeline:**
1. Download / verify datasets
2. Extract video frames
3. Extract MediaPipe hand landmarks
4. Normalise & handle missing landmarks
5. Apply augmentation
6. Generate train / validation / test splits
7. Save `.npy` landmark arrays + label CSVs


In [ ]:
import os, sys, json, csv, shutil, zipfile, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# ── Project root ─────────────────────────────────────────────
ROOT = Path("..")
DATA_RAW        = ROOT / "data" / "raw"
DATA_PROCESSED  = ROOT / "data" / "processed"
DATA_LANDMARKS  = ROOT / "data" / "landmarks"
LABELS_FILE     = ROOT / "labels.json"

for d in [DATA_RAW, DATA_PROCESSED, DATA_LANDMARKS]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths ready ✓")

## 1 — Dataset Sources
Place raw dataset archives inside `data/raw/` before running this notebook.

| Dataset | Type | Classes | Notes |
|---|---|---|---|
| KARSL-502 | Static | 502 | Labels in `KARSL-502_Labels.xlsx` |
| ArASL | Static | 32 | Arabic alphabet + diacritics |
| AUTSL | Dynamic | 226 | Continuous SL, RGB+depth |
| Phoenix-2014T | Dynamic | ~1000 | Sentence-level |


In [ ]:
# ── Load label mapping ────────────────────────────────────────
with open(LABELS_FILE, encoding="utf-8") as f:
    labels = json.load(f)

print(f"Total classes: {len(labels)}")
print("Sample:", dict(list(labels.items())[:5]))

In [ ]:
# ── MediaPipe Hands setup ─────────────────────────────────────
mp_hands = mp.solutions.hands

def extract_landmarks(image_bgr: np.ndarray,
                      static_mode: bool = True) -> np.ndarray | None:
    """
    Extract hand landmarks from a single BGR image.
    Returns flat float32 array of shape (126,) for both hands,
    or None if no hands detected.
    """
    with mp_hands.Hands(
        static_image_mode=static_mode,
        max_num_hands=2,
        min_detection_confidence=0.5,
    ) as hands:
        rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

    if not result.multi_hand_landmarks:
        return None

    all_lm = []
    for hand_lms in result.multi_hand_landmarks:
        for lm in hand_lms.landmark:
            all_lm.extend([lm.x, lm.y, lm.z])

    # Pad to 126 (2 hands × 21 × 3) if only one hand
    while len(all_lm) < 126:
        all_lm.append(0.0)

    return np.array(all_lm[:126], dtype=np.float32)

print("MediaPipe Hands ready ✓")

In [ ]:
# ── Normalisation ─────────────────────────────────────────────
def normalise_landmarks(lm: np.ndarray) -> np.ndarray:
    """
    Translate so wrist (point 0) is at origin, then scale by
    the max extent of the hand span.
    """
    lm = lm.copy().reshape(-1, 3)
    wrist = lm[0].copy()
    lm -= wrist
    scale = np.max(np.abs(lm)) + 1e-6
    lm /= scale
    return lm.flatten()

In [ ]:
# ── Augmentation ──────────────────────────────────────────────
import scipy.ndimage

def augment_landmark_sequence(seq: np.ndarray,
                               noise_std: float = 0.005,
                               rot_max_deg: float = 15.0,
                               scale_range: tuple = (0.85, 1.15),
                               drop_prob: float = 0.1) -> np.ndarray:
    """
    Apply stochastic augmentations to a temporal landmark sequence.
    seq shape: (T, D)
    """
    seq = seq.copy().astype(np.float32)
    T, D = seq.shape

    # 1. Gaussian noise
    seq += np.random.normal(0, noise_std, seq.shape).astype(np.float32)

    # 2. 2-D rotation (x-y plane)
    angle = np.radians(np.random.uniform(-rot_max_deg, rot_max_deg))
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    pts = seq.reshape(T, -1, 3)
    x, y = pts[..., 0].copy(), pts[..., 1].copy()
    pts[..., 0] = cos_a * x - sin_a * y
    pts[..., 1] = sin_a * x + cos_a * y
    seq = pts.reshape(T, D)

    # 3. Scale
    scale = np.random.uniform(*scale_range)
    seq *= scale

    # 4. Translation
    shift = np.random.uniform(-0.05, 0.05, (1, D)).astype(np.float32)
    seq += shift

    # 5. Random frame drop (replace with interpolated)
    mask = np.random.rand(T) < drop_prob
    indices = np.arange(T)
    for i in np.where(mask)[0]:
        neighbors = [j for j in [i-1, i+1] if 0 <= j < T and not mask[j]]
        if neighbors:
            seq[i] = seq[random.choice(neighbors)]

    # 6. Temporal stretching / speed variation
    stretch = np.random.uniform(0.8, 1.2)
    new_len = max(2, int(T * stretch))
    old_idx = np.linspace(0, T - 1, new_len)
    new_seq = np.zeros((new_len, D), dtype=np.float32)
    for d in range(D):
        new_seq[:, d] = np.interp(old_idx, indices, seq[:, d])

    # Re-sample back to original length
    final_idx = np.linspace(0, new_len - 1, T)
    final_seq = np.zeros((T, D), dtype=np.float32)
    for d in range(D):
        final_seq[:, d] = np.interp(final_idx, np.arange(new_len), new_seq[:, d])

    return final_seq

In [ ]:
# ── Process static image dataset ─────────────────────────────
# Expected layout: data/raw/<dataset_name>/<class_label>/<image.jpg>

STATIC_DATASET_DIR = DATA_RAW / "ArASL"   # adjust as needed

records = []   # list of (landmark_array, label)

if STATIC_DATASET_DIR.exists():
    class_dirs = sorted([d for d in STATIC_DATASET_DIR.iterdir() if d.is_dir()])
    label_map = {name: idx for idx, name in enumerate(c.name for c in class_dirs)}

    for cls_dir in tqdm(class_dirs, desc="Classes"):
        label_idx = label_map[cls_dir.name]
        imgs = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png"))
        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            lm = extract_landmarks(img, static_mode=True)
            if lm is not None:
                lm = normalise_landmarks(lm)
                records.append((lm, label_idx))

    print(f"Extracted {len(records)} static samples")
else:
    print(f"[SKIP] {STATIC_DATASET_DIR} not found — place dataset there and re-run")

In [ ]:
# ── Train / Val / Test split and save ────────────────────────
if records:
    random.shuffle(records)
    n = len(records)
    n_train = int(0.80 * n)
    n_val   = int(0.10 * n)

    splits = {
        "train": records[:n_train],
        "val":   records[n_train:n_train + n_val],
        "test":  records[n_train + n_val:],
    }

    for split_name, split_data in splits.items():
        X = np.stack([r[0] for r in split_data])
        y = np.array([r[1] for r in split_data], dtype=np.int64)
        out = DATA_LANDMARKS / "static" / split_name
        out.mkdir(parents=True, exist_ok=True)
        np.save(out / "X.npy", X)
        np.save(out / "y.npy", y)
        print(f"{split_name}: {len(X)} samples, shape {X.shape}")

In [ ]:
# ── Process dynamic video dataset ────────────────────────────
# Expected: data/raw/<dataset>/<class_label>/<video.mp4>
SEQ_LEN = 30   # frames per gesture window
DYNAMIC_DATASET_DIR = DATA_RAW / "AUTSL"   # adjust as needed

dynamic_records = []

if DYNAMIC_DATASET_DIR.exists():
    class_dirs = sorted([d for d in DYNAMIC_DATASET_DIR.iterdir() if d.is_dir()])
    dyn_label_map = {name: idx for idx, name in enumerate(c.name for c in class_dirs)}

    for cls_dir in tqdm(class_dirs, desc="Video classes"):
        label_idx = dyn_label_map[cls_dir.name]
        videos = list(cls_dir.glob("*.mp4")) + list(cls_dir.glob("*.avi"))
        for video_path in videos:
            cap = cv2.VideoCapture(str(video_path))
            frames, lm_list = [], []
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                frames.append(frame)
            cap.release()

            # Sample SEQ_LEN evenly-spaced frames
            if len(frames) < 2:
                continue
            indices = np.linspace(0, len(frames) - 1, SEQ_LEN, dtype=int)
            seq_lm = []
            for i in indices:
                lm = extract_landmarks(frames[i], static_mode=False)
                if lm is None:
                    lm = np.zeros(126, dtype=np.float32)
                seq_lm.append(normalise_landmarks(lm))

            seq = np.stack(seq_lm)  # (SEQ_LEN, 126)
            dynamic_records.append((seq, label_idx))

    print(f"Extracted {len(dynamic_records)} dynamic sequences")
else:
    print(f"[SKIP] {DYNAMIC_DATASET_DIR} not found")

In [ ]:
# ── Augment & save dynamic sequences ─────────────────────────
AUG_FACTOR = 3   # generate N augmented copies per original

if dynamic_records:
    augmented = []
    for seq, label in tqdm(dynamic_records, desc="Augmenting"):
        augmented.append((seq, label))   # original
        for _ in range(AUG_FACTOR):
            augmented.append((augment_landmark_sequence(seq), label))

    random.shuffle(augmented)
    n = len(augmented)
    n_train = int(0.80 * n)
    n_val   = int(0.10 * n)

    splits = {
        "train": augmented[:n_train],
        "val":   augmented[n_train:n_train + n_val],
        "test":  augmented[n_train + n_val:],
    }

    for split_name, split_data in splits.items():
        X = np.stack([r[0] for r in split_data])
        y = np.array([r[1] for r in split_data], dtype=np.int64)
        out = DATA_LANDMARKS / "dynamic" / split_name
        out.mkdir(parents=True, exist_ok=True)
        np.save(out / "X.npy", X)
        np.save(out / "y.npy", y)
        print(f"{split_name}: {X.shape}")

In [ ]:
# ── Dataset statistics visualisation ─────────────────────────
for mode in ["static", "dynamic"]:
    train_path = DATA_LANDMARKS / mode / "train" / "y.npy"
    if not train_path.exists():
        continue
    y = np.load(train_path)
    unique, counts = np.unique(y, return_counts=True)
    plt.figure(figsize=(14, 3))
    plt.bar(unique, counts, color="steelblue")
    plt.title(f"{mode.capitalize()} — class distribution (train)")
    plt.xlabel("Class index")
    plt.ylabel("Samples")
    plt.tight_layout()
    plt.savefig(DATA_PROCESSED / f"{mode}_class_dist.png", dpi=120)
    plt.show()
    print(f"{mode} | classes: {len(unique)} | total: {len(y)} | min: {counts.min()} | max: {counts.max()}")